# COMAVI — Complex-Aware Variant Impact Scoring
### Guided Google Colab notebook for user-supplied missense variants

This notebook runs the public COMAVI workflow from a browser for **any configured hub gene, partner set, and valid missense-variant list**. A GPU is optional: it is used only when you choose the ColabFold structure-prediction route. FoldX itself runs on the CPU and must be supplied under your own licence.

For each variant, COMAVI keeps two products distinct:

1. **Priority score — integrated structural-disruption score (ISDS-v1):** ranks the strength of modeled structural-disruption evidence for follow-up.
2. **Mechanism profile:** preserves the signed monomer-fold, assembled-complex, and partner-binding hypotheses needed to choose an experiment.

The notebook fetches or accepts monomer structures, accepts an experimental or predicted complex, builds a validated YAML configuration, runs the shared COMAVI engine, verifies the output contract, and produces downloadable raw and reader-facing reports.

> **Predict once, score many.** Structures depend on the protein system, not the submitted variant list. After the structures are prepared, edit the variants and rerun Steps 3–4 without predicting the structures again.

Repository: https://github.com/la424/comavi

COMAVI predicts modeled structural disruption and a proposed mechanism. **Neither ISDS-v1 nor the mechanism profile is a pathogenicity verdict.**

## ⚠️ Before you start: supply your own FoldX binary

COMAVI uses [FoldX](https://foldxsuite.crg.eu/) to calculate ΔΔG values. FoldX is **not bundled** with this notebook or the COMAVI repository.

1. Register and download FoldX under the licence appropriate for your use.
2. Download the **Linux** build because Google Colab runs Linux.
3. Upload the `foldx` binary in the next cell.

Without FoldX, you can still parse variants, fetch or upload structures, search the PDB, and build the system configuration, but you cannot perform the energetic calculations.

Structure prediction is a separate step. The most robust route is to upload an AlphaFold Server model. The optional ColabFold route depends on the current Colab GPU environment and may occasionally require upstream fixes.

In [ ]:
#@title Setup: clone the verified COMAVI release and install dependencies { display-mode: "form" }
import os, sys, subprocess, shutil, importlib, re
from pathlib import Path

# Public release after the benchmark and CHD/public-pipeline closeouts.
# Replace with "main" only when deliberately testing unreleased development code.
COMAVI_REF = "aeeaa3956b26edd67115083941954727316ca997"  #@param {type:"string"}

REPO = Path("/content/comavi")
PY = sys.executable

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/la424/comavi.git", str(REPO)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "-q", "--tags", "--prune", "origin"],
        check=True,
    )

ref = COMAVI_REF.strip() or "main"
if ref == "main":
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "--detach", ref], check=True)

resolved_commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

if re.fullmatch(r"[0-9a-fA-F]{40}", ref) and resolved_commit.lower() != ref.lower():
    raise RuntimeError(
        f"Requested COMAVI commit {ref}, but Git resolved {resolved_commit}."
    )

subprocess.run(
    [
        PY,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO / "requirements.txt"),
        "gemmi",
        "matplotlib",
    ],
    check=True,
)

required_runtime_files = [
    REPO / "run.py",
    REPO / "notebooks" / "comavi_helpers.py",
    REPO / "scripts" / "build_isds_variant_report.py",
    REPO / "verification" / "verify_isds_output_surfaces.py",
    REPO / "scripts" / "comavi_v7" / "isds.py",
]
missing_runtime_files = [str(path) for path in required_runtime_files if not path.is_file()]
if missing_runtime_files:
    raise FileNotFoundError(
        "The selected COMAVI ref lacks required public-workflow files:\n  "
        + "\n  ".join(missing_runtime_files)
    )

SCRIPTS = REPO / "scripts"
for directory in (str(REPO / "notebooks"), str(SCRIPTS)):
    if directory not in sys.path:
        sys.path.insert(0, directory)

import comavi_helpers
importlib.reload(comavi_helpers)
H = comavi_helpers

from comavi_v7.isds import ISDS_OUTPUT_COLUMNS, ISDS_VERSION
ISDS_FIELDS = list(ISDS_OUTPUT_COLUMNS)
EXPECTED_ISDS_FIELD_SET = {
    "isds_version",
    "isds_available",
    "isds_v1",
    "isds_energy_ratio_uncapped",
    "isds_energy_component",
    "isds_context_component",
    "isds_dominant_axis",
    "isds_dominant_partner",
    "isds_dominant_signed_ddg",
}
if set(ISDS_FIELDS) != EXPECTED_ISDS_FIELD_SET:
    raise RuntimeError(
        "The selected COMAVI ref exposes an unexpected ISDS output contract: "
        + repr(ISDS_FIELDS)
    )

WORK = Path("/content/work")
STRUCT = WORK / "structures"
STRUCT.mkdir(parents=True, exist_ok=True)
OUT = WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)

gpu = subprocess.run(
    ["bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null || echo none"],
    capture_output=True,
    text=True,
).stdout.strip()

print("COMAVI repository:", REPO)
print("Resolved commit:", resolved_commit)
print("ISDS version:", ISDS_VERSION)
print("ISDS fields:", len(ISDS_FIELDS))
print("GPU:", gpu if gpu and gpu != "none" else "none (needed only for the optional ColabFold route)")
print("Setup OK.")

In [ ]:
#@title Upload your FoldX binary (LINUX build)  { display-mode: "form" }
from google.colab import files
print("Select your LINUX FoldX binary. Colab runs Linux, so a macOS or Windows FoldX build will NOT run here.")
up = files.upload()
fn = list(up.keys())[0]
FOLDX = WORK / "foldx"
shutil.move(fn, FOLDX)
os.chmod(FOLDX, 0o755)
os.environ["FOLDX_BINARY"] = str(FOLDX)

magic = open(FOLDX, "rb").read(4)
ELF = bytes([0x7f, 0x45, 0x4c, 0x46])
MACHO = (bytes([0xcf, 0xfa, 0xed, 0xfe]), bytes([0xce, 0xfa, 0xed, 0xfe]),
         bytes([0xca, 0xfe, 0xba, 0xbe]), bytes([0xfe, 0xed, 0xfa, 0xcf]))
if magic == ELF:
    try:
        subprocess.run([str(FOLDX), "-h"], capture_output=True, text=True, timeout=60)
        print("OK: Linux binary detected and runnable.")
        print("FOLDX_BINARY =", FOLDX)
    except OSError as e:
        print("Binary present but did not execute:", e)
elif magic in MACHO:
    print("ERROR: that is a macOS (Mach-O) FoldX binary - it cannot run on Colab's Linux.")
    print("Download the LINUX build from https://foldxsuite.crg.eu/ and upload that instead, then re-run this cell.")
elif magic[:2] == bytes([0x4d, 0x5a]):
    print("ERROR: that is a Windows (.exe) FoldX binary - it cannot run on Colab's Linux.")
    print("Download the LINUX build from https://foldxsuite.crg.eu/ and upload that instead, then re-run this cell.")
else:
    print("WARNING: this file does not look like a Linux executable (unexpected header).")
    print("Make sure you uploaded the LINUX FoldX binary from https://foldxsuite.crg.eu/.")

---
## Step 1 — Define the protein system and variants

Leave `USE_DEMO` checked for a small two-variant parser and workflow example. Uncheck it to analyze your own proteins.

Variants can be entered in either of two ways:

- **Inline list:** one entry per line as `"GENE AAchange"`, using one-letter amino-acid codes, for example `"SHROOM3 G1003R"`.
- **Upload CSV:** a table containing at least `gene`, `ref_aa`, `position`, and `alt_aa`. Extra columns are preserved.

The notebook explicitly validates those four internal fields before any structural calculation. You may mix variants in the hub and its configured partners.

In [ ]:
#@title Variants and system { display-mode: "both" }
USE_DEMO = True  #@param {type:"boolean"}
VARIANT_INPUT_MODE = "Inline list"  #@param ["Inline list", "Upload CSV"]
HUB_GENE = "SHROOM3"  #@param {type:"string"}
PARTNERS = "ROCK2"  #@param {type:"string"}
MONOMER_SOURCE = "AlphaFold DB (auto-fetch)"  #@param ["AlphaFold DB (auto-fetch)", "Manual upload"]
ORGANISM_ID = 9606  #@param {type:"integer"}

# Enter one variant per line when VARIANT_INPUT_MODE is "Inline list".
VARIANTS = [
    "SHROOM3 G1003R",
    "SHROOM3 T1012N",
]

import pandas as pd
from IPython.display import display, HTML

if USE_DEMO:
    HUB_GENE = "TNNI3"
    PARTNERS = "TNNC1"
    MONOMER_SOURCE = "AlphaFold DB (auto-fetch)"
    VARIANT_INPUT_MODE = "Inline list"
    VARIANTS = ["TNNI3 R162W", "TNNI3 P82S"]
    print(
        "DEMO MODE: TNNI3 + TNNC1 with two distinct substitutions. "
        "Uncheck USE_DEMO for your own genes or upload a CSV."
    )

partner_list = [p.strip() for p in PARTNERS.replace(",", " ").split() if p.strip()]
genes_all = [HUB_GENE] + partner_list

if VARIANT_INPUT_MODE == "Upload CSV" and not USE_DEMO:
    from google.colab import files
    print("Upload a CSV with gene, ref_aa, position, and alt_aa columns.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No variant CSV was uploaded.")
    uploaded_name = next(iter(uploaded))
    vdf = pd.read_csv(uploaded_name, encoding="utf-8-sig")
    vdf.columns = [str(column).strip() for column in vdf.columns]
    verr = []
else:
    vdf, verr = H.parse_variants("\n".join(VARIANTS))

required_variant_columns = ["gene", "ref_aa", "position", "alt_aa"]
missing_variant_columns = [
    column for column in required_variant_columns if column not in vdf.columns
]
if missing_variant_columns:
    raise ValueError(
        "Variant input is missing required internal columns: "
        + ", ".join(missing_variant_columns)
    )

vdf = vdf.copy()
vdf["gene"] = vdf["gene"].astype(str).str.strip().str.lower()
vdf["ref_aa"] = vdf["ref_aa"].astype(str).str.strip().str.upper()
vdf["alt_aa"] = vdf["alt_aa"].astype(str).str.strip().str.upper()
vdf["position"] = pd.to_numeric(vdf["position"], errors="raise").astype(int)

if "variant" not in vdf.columns:
    vdf["variant"] = (
        vdf["ref_aa"]
        + vdf["position"].astype(str)
        + vdf["alt_aa"]
    )
else:
    vdf["variant"] = vdf["variant"].astype(str).str.strip()

if vdf.empty:
    raise ValueError("No valid variants were parsed.")

invalid_aa = vdf.loc[
    ~vdf["ref_aa"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]")
    | ~vdf["alt_aa"].str.fullmatch(r"[ACDEFGHIKLMNPQRSTVWY]")
]
if not invalid_aa.empty:
    raise ValueError(
        "Only standard one-letter amino-acid codes are accepted. Invalid rows:\n"
        + invalid_aa[["gene", "variant"]].to_string(index=False)
    )

print("Hub:", HUB_GENE, "| Partners:", partner_list, "| Monomer source:", MONOMER_SOURCE)
print(f"Parsed {len(vdf)} variants across {vdf['gene'].nunique()} gene(s).")
display(vdf)

if verr:
    print("Skipped inline entries:")
    for line_number, raw, reason in verr:
        print("  line", line_number, repr(raw), "->", reason)

configured_genes = {gene.lower() for gene in genes_all}
unknown_genes = sorted(set(vdf["gene"]) - configured_genes)
if unknown_genes:
    raise ValueError(
        "These variant genes are absent from the hub/partner definition: "
        + ", ".join(unknown_genes)
    )

print("VARIANT INPUT CONTRACT: PASS")

## Step 2a — Monomer structures

Either **auto-fetch** each gene's model from the AlphaFold database (by UniProt), or **upload** your own monomer files (name each file to start with the gene symbol, e.g. `SHROOM3.pdb`).

In [ ]:
#@title Get monomer structures  { display-mode: "form" }
import shutil
gene_uniprot, gene_seq, gene_monomer = {}, {}, {}
try:
    ORGANISM_ID = int(str(ORGANISM_ID).strip())
except (ValueError, TypeError):
    ORGANISM_ID = 9606
    print("ORGANISM_ID was blank/invalid; defaulting to 9606 (human).")

if MONOMER_SOURCE == "AlphaFold DB (auto-fetch)":
    for g in genes_all:
        try:
            acc, cands = H.resolve_uniprot(g, organism_id=ORGANISM_ID)
        except Exception as e:
            print("Could not resolve", g, "on UniProt:", e, "- try Manual upload for this gene.")
            continue
        if not acc:
            print("WARNING: no reviewed UniProt entry for", g, "- use Manual upload for this gene.")
            continue
        try:
            path = H.fetch_alphafold_monomer(acc, STRUCT)
            gene_uniprot[g.lower()] = acc
            gene_monomer[g.lower()] = path
            gene_seq[g.lower()] = H.fetch_uniprot_sequence(acc)
            print(f"{g:10s} {acc}  {H.count_residues(path)} residues  -> {path.name}")
        except Exception as e:
            print("Resolved", g, "to", acc, "but could not fetch its AlphaFold model:", e)
else:
    from google.colab import files
    print("Upload one monomer file per gene (.pdb or .cif), each named to start with the gene symbol.")
    up = files.upload()
    for src in list(up.keys()):
        dest = STRUCT / src
        shutil.move(src, dest)
        stem = dest.stem.lower()
        match = next((g.lower() for g in genes_all if stem.startswith(g.lower())), None)
        if match:
            gene_monomer[match] = dest
            print(f"{match:10s} <- {dest.name}  ({H.count_residues(dest)} residues)")
        else:
            print("Could not map", dest.name, "to a gene; rename to start with the gene symbol.")
    for g in genes_all:
        if g.lower() not in gene_seq:
            try:
                acc, _ = H.resolve_uniprot(g, organism_id=ORGANISM_ID)
                if acc:
                    gene_uniprot[g.lower()] = acc
                    gene_seq[g.lower()] = H.fetch_uniprot_sequence(acc)
            except Exception:
                pass

print("Monomers ready for:", sorted(gene_monomer))

## Step 2b-0 — Check for an experimental complex first

Before predicting a complex, search the PDB for an experimentally determined assembly containing the requested genes. An experimental interface is preferable when it contains the mutated residues and has appropriate chain mapping.

This step is intentionally explicit. After reviewing the search results, enter a PDB identifier and a gene-to-chain map such as `HBB:B,HBA1:A`. COMAVI must know which chain represents each gene; a PDB download without that mapping is not enough.

When no suitable experimental assembly exists, continue to Step 2b and use AlphaFold Server or the optional ColabFold route.

In [ ]:
#@title Search or select an experimental PDB complex { display-mode: "form" }
MAX_HITS = 40  #@param {type:"integer"}
EXPERIMENTAL_PDB_ID = ""  #@param {type:"string"}
EXPERIMENTAL_CHAIN_MAP = ""  #@param {type:"string"}

_need = ("genes_all", "STRUCT")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the setup and variant cells."

hits = H.rcsb_find_and_rank(list(genes_all), max_hits=MAX_HITS, inspect=12)

if not hits:
    print(f"No experimental human complex found covering {', '.join(genes_all)}.")
    print("Continue to Step 2b and provide a predicted complex.")
else:
    print(f"{len(hits)} candidate(s) covering {', '.join(genes_all)}, smallest assembly first:\n")
    print(f"{'PDB':6s} {'chains':>6s} {'res(A)':>7s}  {'year':4s}  method / title")
    for record in hits[:8]:
        resolution = f"{record['resolution_A']:.2f}" if record.get("resolution_A") else "n/a"
        print(
            f"{record['pdb_id']:6s} {str(record['n_chains']):>6s} {resolution:>7s}  "
            f"{record['year']:4s}  {(record['method'] or '')[:18]:18s} {record['title'][:52]}"
        )
    print("\nCheck that the selected entry contains the mutated residues and appropriate biological assembly.")

experimental_complex_pdb = None
experimental_gene_chain = None

if EXPERIMENTAL_PDB_ID.strip():
    pdb_id = EXPERIMENTAL_PDB_ID.strip().upper()
    experimental_complex_pdb = Path(H.fetch_pdb_entry(pdb_id, STRUCT))

    mapping = {}
    for item in EXPERIMENTAL_CHAIN_MAP.split(","):
        item = item.strip()
        if not item:
            continue
        if ":" not in item:
            raise ValueError(
                "EXPERIMENTAL_CHAIN_MAP entries must use GENE:CHAIN, separated by commas."
            )
        gene, chain = [part.strip() for part in item.split(":", 1)]
        mapping[gene.lower()] = chain

    missing_mappings = [gene for gene in genes_all if gene.lower() not in mapping]
    if missing_mappings:
        raise ValueError(
            "The experimental complex was downloaded, but chain mappings are missing for: "
            + ", ".join(missing_mappings)
        )

    experimental_gene_chain = mapping
    print("Experimental complex:", experimental_complex_pdb)
    print("Gene-to-chain map:", experimental_gene_chain)
    print("Step 2b Auto mode will use this experimental complex.")
else:
    print("No experimental PDB selected. Step 2b Auto mode will request a predicted complex.")

## Step 2b — Provide or predict the complex

Choose one route:

- **Auto:** use the experimental PDB selected above when a complete chain map is available; otherwise use the AlphaFold Server upload route.
- **Experimental PDB selected above:** require the downloaded PDB and gene-to-chain map from Step 2b-0.
- **AlphaFold Server:** recommended general route; upload the downloaded `.cif` or `.zip` result.
- **ColabFold GPU:** optional one-click prediction for small or medium complexes. The notebook pins the upstream ColabFold release, but Colab GPU environments can still change.
- **Domain-scoped ColabFold:** predict a specified hub-gene interval while retaining full-length residue numbering through an offset.

### Residue numbering and offsets

Protein structures do not always use the same residue numbering as the submitted variant name. COMAVI defines the mapping as:

`structure position = submitted variant position - offset`

Most systems use zero. Use the two optional override fields in the next cell when a monomer or complex uses a different convention. Enter comma-separated `GENE:OFFSET` pairs, for example `HBB:-1`. A negative offset moves the submitted position forward in the structure; historical HBB E6 numbering maps to residue 7 in the full-length AlphaFold/UniProt monomer, so that monomer uses `HBB:-1`, while mature-chain 2HHB uses zero.

Do not bypass a residue-identity mismatch. The numbering check is designed to prevent a plausible-looking FoldX value from being calculated for the wrong residue.


In [ ]:
#@title Provide or predict the complex structure { display-mode: "form" }
MULTIMER_MODE = "Auto (recommended)"  #@param ["Auto (recommended)", "Experimental PDB selected above", "AlphaFold Server (upload .cif)", "ColabFold (GPU)", "Domain-scoped ColabFold"]
HUB_DOMAIN_RANGE = ""  #@param {type:"string"}
COLABFOLD_RELEASE = "v1.6.1"  #@param {type:"string"}
MONOMER_OFFSET_OVERRIDES = ""  #@param {type:"string"}
MULTIMER_OFFSET_OVERRIDES = ""  #@param {type:"string"}

_need = ("genes_all", "gene_seq", "HUB_GENE")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the preceding cells."

predicted_chain_ids = [chr(ord("A") + index) for index in range(len(genes_all))]
predicted_gene_chain = {
    gene.lower(): predicted_chain_ids[index]
    for index, gene in enumerate(genes_all)
}

hub = HUB_GENE.lower()
mono_off = {gene.lower(): 0 for gene in genes_all}
multi_off = {gene.lower(): 0 for gene in genes_all}

def _parse_offset_overrides(raw_text, label):
    """Parse comma/semicolon/newline-separated GENE:INTEGER pairs."""
    import re as _re

    mapping = {}
    text = str(raw_text or "").strip()
    if not text:
        return mapping

    configured = {gene.lower() for gene in genes_all}
    for item in _re.split(r"[,;\n]+", text):
        item = item.strip()
        if not item:
            continue
        if ":" not in item:
            raise ValueError(
                f"{label} entry {item!r} must use GENE:OFFSET, for example HBB:-1."
            )
        gene, raw_offset = [part.strip() for part in item.split(":", 1)]
        normalized_gene = gene.lower()
        if normalized_gene not in configured:
            raise ValueError(
                f"{label} specifies {gene!r}, which is not in the configured hub/partner set: "
                + ", ".join(genes_all)
            )
        try:
            offset = int(raw_offset)
        except ValueError as error:
            raise ValueError(
                f"{label} offset for {gene!r} must be an integer; received {raw_offset!r}."
            ) from error
        if normalized_gene in mapping and mapping[normalized_gene] != offset:
            raise ValueError(
                f"{label} gives conflicting offsets for {gene!r}: "
                f"{mapping[normalized_gene]} and {offset}."
            )
        mapping[normalized_gene] = offset
    return mapping

manual_monomer_offsets = _parse_offset_overrides(
    MONOMER_OFFSET_OVERRIDES,
    "MONOMER_OFFSET_OVERRIDES",
)
manual_multimer_offsets = _parse_offset_overrides(
    MULTIMER_OFFSET_OVERRIDES,
    "MULTIMER_OFFSET_OVERRIDES",
)

mode = MULTIMER_MODE
if mode == "Auto (recommended)":
    if (
        globals().get("experimental_complex_pdb") is not None
        and globals().get("experimental_gene_chain")
    ):
        mode = "Experimental PDB selected above"
        print("Auto -> experimental PDB selected in Step 2b-0.")
    else:
        mode = "AlphaFold Server (upload .cif)"
        print("Auto -> AlphaFold Server upload route.")

complex_pdb = None
structure_type = "AF"
gene_chain = predicted_gene_chain.copy()
sys_name = "_".join(gene.lower() for gene in genes_all)

if mode == "Experimental PDB selected above":
    complex_pdb = globals().get("experimental_complex_pdb")
    gene_chain = globals().get("experimental_gene_chain") or {}
    if complex_pdb is None or not Path(complex_pdb).is_file():
        raise RuntimeError(
            "No experimental PDB is ready. Return to Step 2b-0, enter a PDB ID and complete chain map, and rerun it."
        )
    missing_mappings = [gene for gene in genes_all if gene.lower() not in gene_chain]
    if missing_mappings:
        raise RuntimeError(
            "Experimental chain mapping is incomplete for: " + ", ".join(missing_mappings)
        )
    structure_type = "PDB"
    complex_pdb = Path(complex_pdb)
    print("Using experimental complex:", complex_pdb)
    print("Gene-to-chain map:", gene_chain)

else:
    sequences = []
    for gene in genes_all:
        sequence = gene_seq.get(gene.lower(), "")
        if not sequence:
            raise RuntimeError(
                f"No sequence is available for {gene}; the predicted-complex routes require one."
            )
        if gene.lower() == hub and HUB_DOMAIN_RANGE.strip():
            start, end = [int(value) for value in HUB_DOMAIN_RANGE.replace("-", " ").split()]
            sequence = H.slice_sequence(sequence, start, end)
            multi_off[hub] = start - 1
            print(
                f"Domain-scoping {HUB_GENE} to {start}-{end} ({len(sequence)} aa); "
                f"multimer offset = {start - 1}."
            )
        sequences.append(sequence)

    total_residues = sum(len(sequence) for sequence in sequences)
    print(f"Total predicted-complex size: approximately {total_residues} residues.")

    if mode == "AlphaFold Server (upload .cif)":
        print("=== AlphaFold Server route ===")
        print("1. Open https://alphafoldserver.com and create one protein entity per chain.")
        print("2. Preserve this chain order:")
        for gene, sequence in zip(genes_all, sequences):
            print(f"   chain {gene_chain[gene.lower()]} = {gene} ({len(sequence)} aa)")
        sequence_file = WORK / f"{sys_name}_sequences.txt"
        sequence_file.write_text(
            "\n".join(f">{gene}\n{sequence}" for gene, sequence in zip(genes_all, sequences)),
            encoding="utf-8",
        )
        print("Sequences saved to:", sequence_file)
        print("3. Download the AlphaFold Server result as .cif or .zip and upload it here.")

        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No AlphaFold Server result was uploaded.")
        source_name = next(iter(uploaded))
        raw_path = WORK / source_name
        shutil.move(source_name, raw_path)

        if raw_path.suffix.lower() == ".zip":
            import zipfile
            with zipfile.ZipFile(raw_path) as archive:
                cif_names = [name for name in archive.namelist() if name.lower().endswith(".cif")]
                if not cif_names:
                    raise RuntimeError("The uploaded ZIP contains no CIF file.")
                archive.extract(cif_names[0], WORK)
                raw_path = WORK / cif_names[0]

        complex_pdb = STRUCT / f"{sys_name}.pdb"
        H.cif_to_pdb(raw_path, complex_pdb)
        print("Converted complex:", complex_pdb, "| residues:", H.count_residues(complex_pdb))

    elif mode in ("ColabFold (GPU)", "Domain-scoped ColabFold"):
        gpu_name = subprocess.run(
            ["bash", "-lc", "nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null"],
            capture_output=True,
            text=True,
        ).stdout.strip()
        if not gpu_name:
            raise RuntimeError(
                "No GPU is attached. In Colab choose Runtime > Change runtime type > GPU, "
                "or use the AlphaFold Server upload route."
            )

        import glob as _glob
        release_token = COLABFOLD_RELEASE.replace("/", "_").replace(".", "_")
        ready = Path(f"/content/COLABFOLD_READY_{release_token}")
        if not ready.exists():
            print("Installing ColabFold", COLABFOLD_RELEASE, "(first run only)...")
            package = (
                "colabfold[alphafold-minus-jax] @ "
                f"git+https://github.com/sokrypton/ColabFold@{COLABFOLD_RELEASE}"
            )
            install = subprocess.run(
                [PY, "-m", "pip", "install", "-q", "--no-warn-conflicts", package],
                capture_output=True,
                text=True,
            )
            if install.returncode != 0:
                raise RuntimeError(
                    "ColabFold installation failed. Use the AlphaFold Server route.\n"
                    + install.stderr[-3500:]
                )
            download = subprocess.run(
                [PY, "-m", "colabfold.download"],
                capture_output=True,
                text=True,
            )
            if download.returncode != 0:
                raise RuntimeError(
                    "ColabFold parameter download failed. Use the AlphaFold Server route.\n"
                    + download.stderr[-3500:]
                )
            ready.touch()
            print("ColabFold installed and model parameters downloaded.")

        fasta = H.write_fasta(sys_name, sequences, WORK / "cf_in")
        cf_out = WORK / "cf_out"
        cf_out.mkdir(exist_ok=True)
        print("Running ColabFold on", fasta.name, "with GPU", gpu_name)
        run = subprocess.run(
            ["colabfold_batch", "--num-models", "1", str(fasta), str(cf_out)],
            capture_output=True,
            text=True,
        )
        if run.stdout.strip():
            print(run.stdout[-2500:])
        if run.returncode != 0:
            raise RuntimeError(
                "ColabFold failed. Use the AlphaFold Server route.\n" + run.stderr[-3500:]
            )

        predicted_models = (
            _glob.glob(str(cf_out / "*relaxed_rank_001*.pdb"))
            or _glob.glob(str(cf_out / "*rank_001*.pdb"))
            or _glob.glob(str(cf_out / "*.pdb"))
        )
        if not predicted_models:
            raise RuntimeError("ColabFold completed but produced no PDB file.")
        complex_pdb = STRUCT / f"{sys_name}.pdb"
        shutil.copy(sorted(predicted_models)[0], complex_pdb)
        print("Complex model:", complex_pdb)

if complex_pdb is None or not Path(complex_pdb).is_file():
    raise RuntimeError("No usable complex structure was produced or selected.")

# Apply manual overrides after any automatic domain-scope offset so an explicit
# user value is the final, auditable mapping used in config.yaml.
mono_off.update(manual_monomer_offsets)
multi_off.update(manual_multimer_offsets)

print("complex_pdb =", complex_pdb)
print("structure_type =", structure_type)
print("gene_chain =", gene_chain)
print("Residue-numbering rule: structure_position = submitted_position - offset")
print("Final monomer offsets:", mono_off)
print("Final multimer offsets:", multi_off)
print("NUMBERING OFFSET CONTRACT: PASS")

In [ ]:
#@title Build and validate the COMAVI configuration { display-mode: "form" }
_need = ("genes_all", "gene_monomer", "complex_pdb", "gene_chain", "structure_type")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the structure cells."

if complex_pdb is None or not Path(complex_pdb).is_file():
    raise RuntimeError("No complex structure is available.")

missing_monomers = [gene for gene in genes_all if gene.lower() not in gene_monomer]
if missing_monomers:
    raise RuntimeError("Missing monomer structures for: " + ", ".join(missing_monomers))

missing_chains = [gene for gene in genes_all if gene.lower() not in gene_chain]
if missing_chains:
    raise RuntimeError("Missing complex-chain assignments for: " + ", ".join(missing_chains))

genes_block = []
for gene in genes_all:
    normalized_gene = gene.lower()
    genes_block.append(
        {
            "gene": normalized_gene,
            "chain": gene_chain[normalized_gene],
            "monomer_file": str(gene_monomer[normalized_gene]),
            "monomer_offset": mono_off.get(normalized_gene, 0),
            "multimer_offset": multi_off.get(normalized_gene, 0),
        }
    )

systems = [
    {
        "name": sys_name,
        "structure_type": structure_type,
        "complex_file": str(complex_pdb),
        "genes": genes_block,
    }
]

cfg_path = H.build_config_yaml(systems, WORK / "config.yaml")
print(cfg_path.read_text(encoding="utf-8"))

try:
    H.validate_config(cfg_path, SCRIPTS)
except Exception as error:
    raise RuntimeError(f"COMAVI configuration validation failed: {error}") from error

print("CONFIGURATION CONTRACT: PASS")

---
## Step 3 — Run COMAVI

This step runs the generic `run.py` entry point against the user-defined YAML configuration and variant table. The raw result retains all submitted columns and adds the complete mechanism profile, structural-context outputs, threshold-specific mechanism calls, and the nine versioned ISDS-v1 fields.

A failed run now stops immediately and preserves stdout and stderr logs in the downloadable bundle.

In [ ]:
#@title Run COMAVI structural assessment { display-mode: "form" }
_need = ("vdf", "FOLDX", "WORK", "cfg_path", "complex_pdb")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the preceding cells."

if not Path(FOLDX).is_file():
    raise FileNotFoundError("The uploaded FoldX binary is missing. Rerun the FoldX upload cell.")

required_variant_columns = ["gene", "ref_aa", "position", "alt_aa"]
missing_variant_columns = [column for column in required_variant_columns if column not in vdf.columns]
if missing_variant_columns:
    raise ValueError("Variant table lost required columns: " + ", ".join(missing_variant_columns))

variant_csv = WORK / "variants.csv"
vdf.to_csv(variant_csv, index=False)
OUT.mkdir(parents=True, exist_ok=True)

environment = dict(os.environ)
environment["FOLDX_BINARY"] = str(FOLDX)
command = [
    PY,
    str(REPO / "run.py"),
    "--config",
    str(cfg_path),
    "--variants",
    str(variant_csv),
    "--structures",
    str(STRUCT),
    "--out",
    str(OUT),
    "--foldx",
    str(FOLDX),
]

print("Running:", " ".join(command))
process = subprocess.run(command, env=environment, capture_output=True, text=True)
(OUT / "run_stdout.log").write_text(process.stdout, encoding="utf-8")
(OUT / "run_stderr.log").write_text(process.stderr, encoding="utf-8")

if process.stdout.strip():
    print(process.stdout[-4000:])
if process.returncode != 0:
    print("STDERR tail:\n", process.stderr[-4000:])
    raise RuntimeError(f"COMAVI exited with status {process.returncode}.")

res_csv = OUT / "structural_results.csv"
if not res_csv.is_file():
    raise FileNotFoundError(f"COMAVI completed without producing {res_csv}.")

print("Raw results:", res_csv)
print("COMAVI RUN: PASS")

In [ ]:
#@title Verify, review, and download the priority score and mechanism profile { display-mode: "form" }
from IPython.display import display, HTML
import json, zipfile
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

_need = ("res_csv", "vdf", "ISDS_FIELDS")
_missing = [name for name in _need if name not in globals()]
assert not _missing, "Undefined: " + ", ".join(_missing) + " — rerun the COMAVI cell."

sdf = pd.read_csv(res_csv, low_memory=False)
if sdf.empty:
    raise RuntimeError("The COMAVI result table is empty.")

required_raw_columns = ["gene", "ref_aa", "position", "alt_aa", "ddg_monomer", "comavi_tier"]
missing_raw_columns = [column for column in required_raw_columns if column not in sdf.columns]
missing_isds_columns = [column for column in ISDS_FIELDS if column not in sdf.columns]
fold_columns = [column for column in sdf.columns if column.startswith("ddg_fold_") and not any(token in column for token in ("_sd", "_runs", "_ci95_", "_vote_"))]
binding_columns = [column for column in sdf.columns if column.startswith("ddg_binding_") and not any(token in column for token in ("_sd", "_runs", "_ci95_", "_vote_"))]
mechanism_columns = [column for column in sdf.columns if column.startswith("comavi_mechanism")]

contract_failures = []
if missing_raw_columns:
    contract_failures.append("missing raw fields: " + ", ".join(missing_raw_columns))
if missing_isds_columns:
    contract_failures.append("missing ISDS-v1 fields: " + ", ".join(missing_isds_columns))
if not fold_columns:
    contract_failures.append("no assembled-complex fold column")
if not binding_columns:
    contract_failures.append("no partner-binding column")
if not mechanism_columns:
    contract_failures.append("no mechanism-call column")

input_variant_ids = set(
    zip(
        vdf["gene"].astype(str).str.lower(),
        vdf["ref_aa"].astype(str).str.upper(),
        vdf["position"].astype(int),
        vdf["alt_aa"].astype(str).str.upper(),
    )
)
output_variant_ids = set(
    zip(
        sdf["gene"].astype(str).str.lower(),
        sdf["ref_aa"].astype(str).str.upper(),
        pd.to_numeric(sdf["position"], errors="raise").astype(int),
        sdf["alt_aa"].astype(str).str.upper(),
    )
)
missing_variants = sorted(input_variant_ids - output_variant_ids)
if missing_variants:
    contract_failures.append(f"submitted variants absent from output: {missing_variants}")

if contract_failures:
    raise RuntimeError("ARBITRARY-VARIANT OUTPUT CONTRACT: FAIL\n  " + "\n  ".join(contract_failures))

print(f"Rows: {len(sdf)} | submitted variants represented: {len(input_variant_ids)}/{len(input_variant_ids)}")
print(f"Assembled-complex columns: {len(fold_columns)} | binding columns: {len(binding_columns)}")
print("ISDS version values:", sorted(set(sdf["isds_version"].dropna().astype(str))))
print("ARBITRARY-VARIANT OUTPUT CONTRACT: PASS")

summary = H.per_partner_table(sdf)
display(HTML(H.render_cards_html(sdf)))
print("Per-variant × partner mechanism summary:")
display(summary)

available = sdf["isds_available"].fillna(False).astype(str).str.lower().isin({"true", "1", "1.0", "yes"})
priority_columns = [
    column
    for column in (
        "gene",
        "variant",
        "system",
        "isds_available",
        "isds_v1",
        "isds_energy_component",
        "isds_context_component",
        "isds_dominant_axis",
        "isds_dominant_partner",
        "isds_dominant_signed_ddg",
        "comavi_tier",
        "comavi_mechanism_t10",
        "comavi_mechanism_t25",
        "comavi_mechanism",
    )
    if column in sdf.columns
]
priority_table = sdf.loc[available, priority_columns].copy()
if "isds_v1" in priority_table.columns:
    priority_table = priority_table.sort_values("isds_v1", ascending=False, kind="mergesort")

print("Priority score table (ISDS-v1 is a prioritization index, not a probability):")
if priority_table.empty:
    print("No row had sufficient evaluable energy and tier information for ISDS-v1.")
else:
    display(priority_table)

plot_frame = summary.copy()
plot_frame["label"] = plot_frame["gene"].str.upper() + " " + plot_frame["variant"]
axis_plot = plot_frame.groupby("label").agg(
    monomer=("ddg_monomer", "first"),
    assembled_complex=("ddg_fold", lambda values: np.nanmax(np.abs(values)) if values.notna().any() else np.nan),
    binding=("ddg_binding", lambda values: np.nanmax(np.abs(values)) if values.notna().any() else np.nan),
)
axis = axis_plot.plot(kind="barh", figsize=(7, 0.5 * len(axis_plot) + 1))
axis.set_xlabel("maximum |ΔΔG| (kcal/mol)")
axis.set_title("Per-variant structural effect by modeled axis")
plt.tight_layout()
axis_png = OUT / "per_variant_axes.png"
plt.savefig(axis_png, dpi=140)
plt.show()

mechanism_summary_csv = OUT / "comavi_mechanism_summary.csv"
summary.to_csv(mechanism_summary_csv, index=False)

report_dir = OUT / "public_report"
report_dir.mkdir(parents=True, exist_ok=True)
report_prefix = "comavi_variants"
report_command = [
    PY,
    str(REPO / "scripts" / "build_isds_variant_report.py"),
    str(res_csv),
    "--out-dir",
    str(report_dir),
    "--prefix",
    report_prefix,
    "--top-n",
    "50",
]
report_process = subprocess.run(report_command, capture_output=True, text=True)
if report_process.stdout.strip():
    print(report_process.stdout[-2500:])
if report_process.returncode != 0:
    raise RuntimeError("Public report generation failed:\n" + report_process.stderr[-3500:])

full_report_csv = report_dir / f"{report_prefix}_with_isds_v1.csv"
prioritized_csv = report_dir / f"{report_prefix}_prioritized.csv"
summary_json = report_dir / f"{report_prefix}_isds_summary.json"
markdown_report = report_dir / f"{report_prefix}_isds_report.md"

verify_command = [
    PY,
    str(REPO / "verification" / "verify_isds_output_surfaces.py"),
    str(full_report_csv),
]
verify_process = subprocess.run(verify_command, capture_output=True, text=True)
print(verify_process.stdout.strip())
if verify_process.returncode != 0:
    raise RuntimeError("ISDS output verification failed:\n" + verify_process.stderr[-3500:])

report_summary = json.loads(summary_json.read_text(encoding="utf-8"))
print("Report summary:")
print(json.dumps(report_summary, indent=2))

release_record = OUT / "COMAVI_run_provenance.txt"
release_record.write_text(
    "\n".join(
        [
            f"comavi_commit={resolved_commit}",
            f"isds_version={ISDS_VERSION}",
            f"input_variants={len(vdf)}",
            f"output_rows={len(sdf)}",
            f"isds_available_rows={int(available.sum())}",
            "monomer_offsets=" + json.dumps(mono_off, sort_keys=True, separators=(",", ":")),
            "multimer_offsets=" + json.dumps(multi_off, sort_keys=True, separators=(",", ":")),
            f"complex_structure_type={structure_type}",
            f"complex_structure={Path(complex_pdb).name}",
            "gene_chain_map=" + json.dumps(gene_chain, sort_keys=True, separators=(",", ":")),
        ]
    )
    + "\n",
    encoding="utf-8",
)

bundle_path = WORK / "COMAVI_variant_results_bundle.zip"
bundle_files = [
    res_csv,
    mechanism_summary_csv,
    full_report_csv,
    prioritized_csv,
    summary_json,
    markdown_report,
    axis_png,
    cfg_path,
    WORK / "variants.csv",
    OUT / "run_stdout.log",
    OUT / "run_stderr.log",
    release_record,
]
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in bundle_files:
        if Path(path).is_file():
            archive.write(path, arcname=Path(path).name)

from google.colab import files
print("Downloading verified result bundle:", bundle_path)
files.download(str(bundle_path))

---
## Step 4 (optional) — Four-way concordance

Overlay your **manually-curated** AlphaMissense + Franklin/ClinVar calls next to the structural result. This is the only step that needs hand-entered data; skip it if you only want the structural assessment.

In [ ]:
#@title (Optional) Build the concordance template  { display-mode: "form" }
ctpl = H.concordance_template(vdf)
ctpl_path = OUT / "concordance_template.csv"
ctpl.to_csv(ctpl_path, index=False)
print("Fill the three columns (AM pathogenicity, AM class, franklin) per variant.")
print("Option A: edit ANNOTATIONS inline in the next cell. Option B: edit this CSV and upload it.")
display(ctpl)
from google.colab import files as _f
_f.download(str(ctpl_path))

In [ ]:
#@title (Optional) Run four-way concordance { display-mode: "form" }
ANNOTATION_MODE = "Upload filled template"  #@param ["Upload filled template", "Use ANNOTATIONS dict below"]

ANNOTATIONS = [
    # {"gene":"shroom3", "variant":"G1003R", "AM pathogenicity":0.81,
    #  "AM class":"likely_pathogenic", "franklin":"Likely Pathogenic"},
]

if ANNOTATION_MODE == "Upload filled template":
    from google.colab import files
    print("Upload a filled CSV/XLSX with gene, variant, AM pathogenicity, AM class, and franklin columns.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No annotation file was uploaded.")
    source_name = next(iter(uploaded))
    annotations = (
        pd.read_excel(source_name)
        if source_name.lower().endswith((".xlsx", ".xls"))
        else pd.read_csv(source_name)
    )
else:
    annotations = pd.DataFrame(ANNOTATIONS)

if annotations.empty:
    print("No annotations provided — skipping concordance.")
else:
    master, concordance_process = H.run_concordance(
        res_csv,
        annotations,
        OUT / "concordance",
        SCRIPTS,
        python_exe=PY,
    )
    print(concordance_process.stdout[-1500:])
    if concordance_process.returncode != 0:
        raise RuntimeError("Concordance failed:\n" + concordance_process.stderr[-2500:])
    if master and master.exists():
        concordance = pd.read_csv(master, low_memory=False)
        keep = [
            column
            for column in (
                "gene",
                "variant",
                "isds_available",
                "isds_v1",
                "isds_dominant_axis",
                "comavi_tier",
                "comavi_mechanism_t10",
                "comavi_mechanism_t25",
                "comavi_mechanism",
                "AM pathogenicity",
                "AM class",
                "franklin",
            )
            if column in concordance.columns
        ]
        display(concordance[keep])
        from google.colab import files as download_files
        download_files.download(str(master))
    else:
        raise RuntimeError("Concordance completed without an output file.")

---
## Notes, limits, and troubleshooting

- **FoldX is required and not bundled.** Upload a licensed Linux build.
- **Current COMAVI code:** the setup cell pins the verified public merge commit and prints the exact resolved commit into the notebook and result bundle.
- **Other variants and genes:** the notebook uses the generic `run.py` entry point. A new gene system requires a valid hub/partner definition, monomer structures, a mapped complex, and variants with `gene`, `ref_aa`, `position`, and `alt_aa`.
- **Two outputs, two uses:** use ISDS-v1 to prioritize follow-up; use the signed mechanism profile to decide whether stability, assembly, or binding should be tested.
- **ISDS-v1 is not a probability.** It can be unavailable when no energetic axis or tier is evaluable. A silent or unavailable result does not establish benignity.
- **GPU is optional.** It is needed only for the ColabFold route. AlphaFold Server upload works without installing ColabFold in the current runtime.
- **ColabFold is upstream software.** This notebook pins a release, but Google Colab images and GPU libraries can still change. If installation or prediction fails, use the AlphaFold Server route and upload its CIF output.
- **Large complexes:** use AlphaFold Server or domain-scoped prediction rather than a free Colab GPU.
- **Experimental complexes:** supply the PDB ID and a complete gene-to-chain map. Confirm that the deposited construct contains every mutated residue.
- **Residue numbering:** use `MONOMER_OFFSET_OVERRIDES` and `MULTIMER_OFFSET_OVERRIDES` when structure numbering differs from the submitted variant convention. COMAVI uses `structure_position = submitted_position - offset` and will stop on an amino-acid mismatch rather than score the wrong site.
- **Predict once, score many:** edit the variant list and rerun Steps 3–4; the structures can be reused.
- **Pathogenicity remains separate:** the optional concordance overlays external evidence, but COMAVI itself reports structural disruption, priority, and mechanism.

Repository: https://github.com/la424/comavi